# Testing the GCNN with Visual Feature Inputs

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from image_processing_tools.util.load_files import find_files_by_pattern
from image_processing_tools.image_class.image_container import ImageContainer
from pathlib import Path
import matplotlib
# matplotlib.use('Agg') # Critical: prevents plotting to screen/RAM

search_path = (
    "~/A8/Data_Eva/Training data Chuyao/MONO culture in DMEM + 10% FBS/CET161_ASH1Q670_02_CY5, DAPI",
    "~/A8/Data_Eva/Training data Chuyao/MONO culture in DMEM + 10% FBS/CET155_ASH1Q670_04_CY5, DAPI",
    "~/A8/Data_Eva/Training data Chuyao/MONO culture in DMEM + 10% FBS/CET147_ECE1Q670_06_CY5, DAPI",
    "~/A8/Data_Eva/Training data Chuyao/MONO culture in DMEM + 10% FBS/CET146_ASH1Q670_07_CY5, DAPI",
    "~/A8/Data_Eva/Training data Chuyao/MONO culture in DMEM + 10% FBS/CET146_ASH1Q670_02_CY5, DAPI",
    "~/A8/Data_Eva/Training data Chuyao/MONO culture in DMEM + 10% FBS/CET145_ASH1Q670_02_CY5, DAPI",
    )
file_pattern = ("CROP_*.tif",
                "MAX_*",
                "SHARPEST*C2*.tif",
                "*DIC*.tif",
                "MAX_CET145*")

config = {
    "preprocessing": {
        "resize_image": False,
        "max_dim": 1080,
        "outlier_percentile": 0.35,
        "quantization": "16bit",
        "correct_DIC_shift": [5,22]
    }
}

# Find the files. The 'files' variable will hold the list of Path objects.
dapi_files = []
dic_files = []
for p in search_path: 
    dapi_files.append(find_files_by_pattern(p, file_pattern[2], verbose=True))
    dic_files.append(find_files_by_pattern(p, file_pattern[3], verbose=True))

dapi_imgs = [ImageContainer(img,config) for img in dapi_files]
dic_imgs = [ImageContainer(img,config) for img in dic_files]
dapi_dic_imgs = [dapi+dic for dapi,dic in zip(dapi_imgs,dic_imgs)]

## Detect nuclei and extract graphs

In [ ]:
from image_processing_tools.dapi_tracing.nuclei_detection import detect_nuclei_rf, filter_nuclei_xgb
from image_processing_tools.dapi_tracing.extract_graph import extract_graph
from image_processing_tools.rf_nuclei.rf_load_models import load_model

dapi_imgs_merged = [img.merge() for img in dapi_imgs]
dapi_dic_imgs_merged = [img.merge() for img in dapi_dic_imgs]

rf_model = load_model(Path('~/A8/Data_Eva/Training data Chuyao/MONO culture in DMEM + 10% FBS/batch_rf_model.joblib').expanduser())

def extract_graph_pipeline_rf(seg_img, model):
    labels, _ = detect_nuclei_rf(seg_img, model)
    valid_nuclei_data, _, _, clean_labels = filter_nuclei_xgb(labels)
    nuclei_df, nuclei_centroids, paths_df, edge_index = extract_graph(
        valid_nuclei_data, seg_img, clean_labels > 0, show_plot=True, show_edge=False
    )
    return nuclei_df, nuclei_centroids, paths_df, edge_index

nuclei_df_list, paths_df_list, edge_index_list, centroid_list = [], [], [], []
for i, img in enumerate(dapi_imgs_merged):
    nuclei_df, nuclei_centroids, paths_df, edge_index = extract_graph_pipeline_rf(img, rf_model)
    nuclei_df_list.append(nuclei_df)
    centroid_list.append(nuclei_centroids)
    paths_df_list.append(paths_df)
    edge_index_list.append(edge_index)

In [ ]:
# import pandas as pd
# # Preview edge indices for each graph to cross-reference when labeling.
# for i, ei in enumerate(edge_index_list):
#     print(f"Graph {i}: {len(ei[0])} edges")
#     display(pd.DataFrame(ei))

## Label edges (manual)

For each graph, list only the `(source_node, target_node)` pairs that are
TRUE hyphal connections. Every other candidate edge is auto-labeled 0 inside
`create_pyg_data`. Matching is undirected so `(u, v)` and `(v, u)` are equivalent.

In [ ]:
edge_labels_list = [
    # Graph 0: list of (u, v) for TRUE edges
    [(0,1),(1,2),(1,3),(2,4)],
    # Graph 1
    [(0,1),(1,2),(1,5),(3,4),(4,5)],
    # Graph 2
    [(2,4),(0,1),(1,3)],
    # Graph 3
    [(11,12),(12,13),(13,14),(10,11),(9,11),(8,10),(6,7),(5,7),(4,5),(3,4),(2,3),(1,3),(0,1)],
    # Graph 4
    [(0,1),(1,2),(2,7),(7,9),(3,7),(4,5),(4,6),(6,8),(8,10),(10,11)],
    # Graph 5
    [(2,4),(4,6),(6,7),(8,10),(10,12),(9,12),(12,14),(14,15),(13,14),(11,13)],
]

## Build PyG dataset with microsam visual features

Microsam feature maps must already be precomputed for these images (see
`Get microsam Features.ipynb`). The paths below are derived from each
`ImageContainer`'s source path and name, matching the save convention in
`compute_microsam_features`.

In [ ]:
microsam_paths = [
    img._source_paths[0].parent / f"DAPI_DIC_microsam_features.npz"
    for img in dapi_imgs
]

# Sanity check
for p in microsam_paths:
    assert p.exists(), f"Missing microsam feature file: {p}"
print(f"Found {len(microsam_paths)} microsam feature files.")

In [ ]:
from image_processing_tools.dapi_tracing.gnn_data import create_pyg_data, save_pyg_dataset, load_pyg_dataset, StripHeavyAttrs

DATASET_ROOT = '/home/wangchuyao/Projects/C_Albicans Thesis Project/7. Data/graph_datasets/visual'

pyg_data_list = create_pyg_data(
    edge_indices=edge_index_list,
    nuclei_features_list=nuclei_df_list,
    path_features_list=paths_df_list,
    edge_labels_list=edge_labels_list,
    images_list=dapi_dic_imgs_merged,
    centroids_list=centroid_list,
    microsam_paths_list=microsam_paths,
    max_edge_length_neg=10.0,  # drop negative candidate edges > 10 nucleus lengths
)
save_pyg_dataset(pyg_data_list, DATASET_ROOT)

# After the first run, switch to the line below to skip rebuild:
pyg_data_list = load_pyg_dataset(DATASET_ROOT)
# Use when training without visual layers
# strip = StripHeavyAttrs(keys=('microsam_embedding', 'pixels_per_feature'))  # keep 'image' for plots                                                                                            
# pyg_data_list = [strip(d) for d in pyg_data_list]     
# May see some warning, harmless

## Run cross-validation with visual features

In [ ]:
from image_processing_tools.dapi_tracing.gnn_train import n_fold_validation

model_params = {
    'hidden_channels': 128,
    'dropout_p': 0.5,
    'use_visual_features': True,
    'd_visual': 32,             # sweep this; visual-feature dimension added to raw node/edge features
    'node_box_size': 150,       # pixels (square) around each nucleus centroid
    'edge_box_margin_frac': 0.15,
    'edge_box_margin_floor': 20,
    'roi_output_size': 7,
}

for i in range(3):
    print(f'Starting repeated CV {i}')
    results = n_fold_validation(
        dataset=pyg_data_list,
        num_folds=6,
        max_epochs=500,
        batch_size=5,
        learning_rate=0.1,
        model_params=model_params,
        patience=100,
        degree_penalty_weight=0,
        neg_sample_ratio=1.0,
        experiment=f"visual_dVisual{model_params['d_visual']}_h{model_params['hidden_channels']}_fixDICShift_trimNeg10_noDegLoss_minEpochFloor_swapNorm_labelSmooth_interpretExpand/repeat_{i}",
        min_epoch=50,
        label_smoothing=0.1,
        log_train_embeddings=True
    )

In [ ]:
from image_processing_tools.dapi_tracing.summarize_cv_logs import summarize_cv_logs, pivot_by_fold
df = summarize_cv_logs("output/cv_experiment/visual/visual_dVisual32_h128_fixDICShift_trimNeg10_noDegLoss_minEpochFloor_swapNorm/")

In [ ]:
pivot_by_fold(df)

## Overfit test

In [ ]:
# from image_processing_tools.dapi_tracing.gnn_train import train_overfit_test

# model_params = {
#     'hidden_channels': 64,
#     'dropout_p': 0.5,
#     'use_visual_features': True,
#     'd_visual': 32,             # sweep this; visual-feature dimension added to raw node/edge features
#     'node_box_size': 150,       # pixels (square) around each nucleus centroid
#     'edge_box_margin_frac': 0.15,
#     'edge_box_margin_floor': 20,
#     'roi_output_size': 7,
# }

# results = train_overfit_test(
#     dataset=pyg_data_list,
#     max_epochs=500,
#     batch_size=6,
#     learning_rate=0.1,
#     model_params=model_params,
#     patience=100,
#     degree_penalty_weight=2.0,
#     neg_sample_ratio=1.0,
#     experiment=f"visual_overfit_dVisual{model_params['d_visual']}_h{model_params['hidden_channels']}",
# )